# Installation
install torch for you GPU/MPS/CPU https://pytorch.org/get-started/locally/



This example code can export from DICOM to a BIDS-Nifty dataset. Loop over the dataset and segment all images. It find and merge leg images (e. g. three views). It will randomly select one with pelvis to feet present and applies TREG leg on it.

In [ ]:
%load_ext autoreload
%autoreload 2
try:
    import deepali
    import TPTBox
except Exception:
    %pip install TPTBox
    %pip install hf-deepali
    %pip install nnunetv2

    import deepali
    import TPTBox
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from TPTBox import NII, POI_Global, to_nii
from TPTBox.core.vert_constants import Full_Body_Instance

from treg.angle import compute_angles
from treg.basics import resolve_device


In [ ]:
dataset = "/media/data/robert/code/TReg/private/dataset-treg"
folder = "rawdata"
dicom_folder_in = None #Path("/media/data/robert/code/TReg/private/Pat0001")

dataset_id = 12
ddevice: Literal["cpu", "cuda", "mps"] = "cuda"
gpu_id = 0  # Only used for cuda

run_only_one = True
subject_name = None


In [ ]:
# Atlas
atlas_seg_file: str | Path = "data/leg/sub-atlas_seg-VIBESeg-12_msk.nii.gz"  # default is a left leg
atlas_file: str | Path = "data/leg/sub-atlas_seg-poi_poi.json"
atlas_seg_subdivided_file: str | Path | None = "data/leg/sub-atlas_seg-subregion_msk.nii.gz"


In [ ]:
from TPTBox.core.dicom.dicom_extract import extract_dicom_folder

# TODO Step 0 export to a nii dataset
if dicom_folder_in is not None:
    # Uncomment and change paths
    extract_dicom_folder(dicom_folder=dicom_folder_in, dataset_path_out=Path(dataset), use_session=True)

In [ ]:
import numpy as np
from TPTBox import BIDS_FILE, BIDS_Global_info
from TPTBox.segmentation import run_vibeseg
from TPTBox.stitching import stitching

# -----------------------------------------------------------------------------
# Loop over the dataset an merge images with the same kernal. Segment all images.
# The dataset must be in the BIDS-Format and the dicom header must be provided as a json with the same file name and path. extract_dicom_folder creates such a dataset
# Will select best complete Leg from a 3 view stich (pelvic, knee, foot). Ignores type of recon kernal.
# -----------------------------------------------------------------------------
bgi = BIDS_Global_info(dataset, parents=folder)

# Dictionary storing the final selected image/segmentation per subject
subs = {}

# -----------------------------------------------------------------------------
# Iterate through all subjects in the BIDS dataset
#
# sub:
#     subject ID string
#
# subj:
#     subject-specific BIDS object
# -----------------------------------------------------------------------------
for sub, subj in bgi.iter_subjects(sort=True):
    # Create a query object to search files belonging to this subject
    q = subj.new_query(flatten=True)

    # Keep only CT images
    q.filter_format("ct")

    # Keep only NIfTI files
    q.filter_filetype("nii.gz")

    # Optional:
    # Restrict to isotropic acquisitions only
    # q.filter("acq","iso",required=False)

    # Ignore localizer scans
    # (localizer are low-quality planning scans)
    q.filter("part", lambda x: x != "localizer", required=False)

    # -------------------------------------------------------------------------
    # Dictionaries grouping images by acquisition/kernel properties
    #
    # img:
    #     stores original image files
    #
    # segs:
    #     stores segmentation outputs
    # -------------------------------------------------------------------------
    img: dict[str, list[BIDS_FILE]] = {}
    segs: dict[str, list] = {}

    # -------------------------------------------------------------------------
    # Loop over all matching CT files
    # -------------------------------------------------------------------------
    for file in q.loop_list(sort=True):
        # Create output segmentation path
        # Example:
        # sub-001_seg-VIBESeg-XYZ_msk.nii.gz
        out_file = file.get_changed_path("nii.gz", "msk", info={"seg": f"VIBESeg-{dataset_id}"})
        # ---------------------------------------------------------------------
        # Run automatic segmentation
        #
        # gpu=0:
        #     use GPU index 0
        #
        # ddevice="cuda":
        #     inference on CUDA GPU
        #
        # dataset_id:
        #     segmentation model identifier
        # ---------------------------------------------------------------------
        out = run_vibeseg(file, out_file, gpu=gpu_id, ddevice=ddevice, dataset_id=dataset_id)
        # Load accompanying JSON metadata
        j: dict = file.open_json()
        # ---------------------------------------------------------------------
        # Build a grouping key ("kernel")
        #
        # Images with identical:
        # - convolution kernel
        # - session
        # - acquisition
        # - part
        #
        # are stitched together later.
        # ---------------------------------------------------------------------
        kernel = j.get("ConvolutionKernel", "ct")
        kernel += "-" + str(file.get("ses", ""))
        kernel += "-" + str(file.get("acq", ""))
        kernel += "-" + str(file.get("part", ""))

        # Initialize lists if key does not exist yet
        if kernel not in img:
            img[kernel] = []
            segs[kernel] = []
        # Store image and segmentation
        img[kernel].append(file)
        segs[kernel].append(out)

    # -------------------------------------------------------------------------
    # Stitch images belonging to the same acquisition group
    # -------------------------------------------------------------------------
    for name, l_images in img.items():
        # Nothing to stitch if only one image exists
        if len(l_images) <= 1:
            continue

        # Build sequence identifier from all sequence names
        seq = "-".join(sorted([str(a.get("sequ")) for a in l_images]))

        # Output stitched CT image
        out = l_images[0].get_changed_path("nii.gz", "msk", "rawdata", info={"sequ": f"stiched-{seq}"})
        # Output stitched segmentation
        out_seg = l_images[0].get_changed_path("nii.gz", "msk", info={"sequ": f"stiched-{seq}", "seg": f"VIBESeg-{dataset_id}"})
        # Output blending/ramp mask
        #
        # The ramp image stores blending weights used during stitching.
        out_ramp = l_images[0].get_changed_path(
            "nii.gz", "ramp", info={"sequ": f"stiched-{seq}", "seg": "ramp"}, non_strict_mode=True
        )
        # ---------------------------------------------------------------------
        # Stitch original CT volumes
        # ---------------------------------------------------------------------
        if not out.exists():
            stitching(l_images, out, is_ct=True, verbose=True, verbose_stitching=True, dtype=np.int16, store_ramp=True, ramp_path=out_ramp)
        # ---------------------------------------------------------------------
        # Stitch segmentation masks
        # ---------------------------------------------------------------------
        if not out_seg.exists():
            stitching(segs[name], out_seg, is_seg=True, verbose=True, verbose_stitching=True)
            # Compress datatype to smallest possible unsigned integer
            # to reduce file size
            to_nii(out_seg, True).set_dtype("smallest_uint").save(out_seg)
        # ---------------------------------------------------------------------
        # Select the "best" acquisition per subject
        #
        # Priority:
        #   1. isotropic ("iso")
        #   2. axial ("ax")
        #   3. everything else
        #
        # Additionally:
        #   only keep cases where labels 11-14 exist
        # ---------------------------------------------------------------------
        if (
            sub not in subs
            or (l_images[0].get("acq", "") == "iso" and subs[sub]["acq"] != "iso")
            or (l_images[0].get("acq", "") == "ax" and subs[sub]["acq"] not in ["iso", "ax"])
        ):
            # Get all labels present in segmentation
            u = to_nii(out_seg, True).unique()
            # Require labels 11,12,13,14 to exist
            # version 12 may additionally require label 100
            if all(a in u for a in range(11, 15)) or all(a in u for a in range(111, 115)):
                # Store final selected subject entry
                subs[sub] = {
                    "img": out,  # stitched CT image
                    "seg": out_seg,  # stitched segmentation
                    "dataset": l_images[0].dataset,  # originating dataset
                    "bin_msk": out_ramp,  # stitching ramp mask
                    "acq": l_images[0].get("acq", ""),  # acquisition type
                }

In [ ]:
subject_names = ([next(iter(subs.keys()))] if subject_name is None else [subject_name]) if run_only_one else list(subs.keys())
print(subject_names)

In [ ]:
#############################################
################# Input #####################
#############################################
all_tasks = []
for subject_name in subject_names:
    files = subs[subject_name]
    # {"img":out,"seg":out_seg,"dataset":l[0].dataset}
    ds = BIDS_FILE(files["img"], files["dataset"])
    sides = ["left", "right"]
    for side in sides:
        files[side] = {}
        files[side]["target_out_poi"] = ds.get_changed_path("json", "poi", info={"desc": "atlas", "seg": side})
        files[side]["target_out_subdivided"] = ds.get_changed_path(
            "nii.gz", "msk", info={"desc": "atlas", "seg": side}, additional_folder=side
        )
        files[side]["target_out_angle"] = ds.get_changed_path("nii.gz", "msk", info={"desc": "angle", "seg": side}, additional_folder=side)
        files[side]["target_out_angle2"] = ds.get_changed_path("nii.gz", "msk", info={"desc": "angle-veerman", "seg": side})
        files[side]["mirror"] = "right" in side

## Generate Segmentation


In [ ]:
from treg.basics import run_all

run_all(files, sides, ddevice=ddevice, gpu=gpu_id)